# Eksplorasi: RAG Pipeline dengan Function Wrapper

Notebook ini berisi versi lanjutan dari notebook 03. Alur RAG sudah dibungkus ke dalam fungsi `tanyakan()` agar mudah dipakai ulang sebelum dipindahkan ke `src/` atau dashboard.

In [2]:
import os
import re
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

# Memuat variabel lingkungan (.env)
load_dotenv('../.env')

# 1. Inisialisasi Model LLM (Chat) dan Embedding
embeddings = OpenAIEmbeddings(
    openai_api_key=os.getenv("SUMOPOD_API_KEY"),
    openai_api_base=os.getenv("SUMOPOD_API_BASE"),
    model="text-embedding-3-small"
)

llm = ChatOpenAI(
    openai_api_key=os.getenv("SUMOPOD_API_KEY"),
    openai_api_base=os.getenv("SUMOPOD_API_BASE"),
    model="glm-5-turbo", # Model GLM lebih cepat dari SumoPod
    temperature=0.0 # Temperature 0 agar AI menjawab secara faktual, bukan kreatif/mengarang
)
print("LLM & Embedding berhasil diinisialisasi.")

LLM & Embedding berhasil diinisialisasi.


### 1. Memuat Ulang Database Vector (ChromaDB)
Kita tidak perlu lagi membaca PDF dari awal, cukup load database yang ada di folder `vector_store`.

In [3]:
persist_directory = "../vector_store"

# Memuat ChromaDB dari disk lokal
vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

# Membuat objek 'retriever' yang akan mengambil 2 teks paling mirip agar respons lebih cepat
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Database lokal berhasil dihubungkan.")

Database lokal berhasil dihubungkan.


In [4]:
# Cek jumlah halaman yang berhasil disimpan di Vector Store
all_data = vectorstore.get()
halaman_unik = set()

for metadata in all_data['metadatas']:
    if metadata and 'page' in metadata:
        halaman_unik.add(metadata['page'])

halaman_terurut = sorted(list(halaman_unik))

print(f"Total chunk (potongan teks) di database: {len(all_data['ids'])}")
print(f"Total halaman yang di-vektorisasi: {len(halaman_unik)} halaman")
if len(halaman_terurut) > 0:
    print(f"Halaman yang tercakup: {halaman_terurut[0]} sampai {halaman_terurut[-1]}")


Total chunk (potongan teks) di database: 975
Total halaman yang di-vektorisasi: 315 halaman
Halaman yang tercakup: 0 sampai 320


### 2. Menyusun Prompt (Instruksi Sistem)
Kita memberikan instruksi ketat agar LLM bertindak sebagai asisten dokumen.

In [5]:
system_prompt = (
    "Anda adalah asisten cerdas untuk tugas menjawab pertanyaan berdasarkan dokumen.\n"
    "Gunakan potongan konteks yang diambil berikut ini untuk menjawab pertanyaan.\n"
    "Jika Anda tidak tahu jawabannya, katakan saja bahwa Anda tidak tahu, jangan mencoba mengarang jawaban.\n"
    "Jaga agar jawaban tetap ringkas, padat, dan jelas (maksimal 3 paragraph 5 kalimat jika memungkinkan).\n"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

### 3. Membangun RAG Pipeline sebagai Fungsi

In [6]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def detect_page_request(question):
    match = re.search(r"\bhal(?:aman|amaan)?\.?\s*(\d+)\b", question.lower())
    if not match:
        return None
    page_number = int(match.group(1))
    return page_number if page_number >= 1 else None

def get_page_chunks(page_number):
    # Metadata page dari PDF loader dimulai dari 0, jadi halaman 10 = page index 9.
    page_index = page_number - 1
    results = vectorstore.get(where={"page": page_index})
    documents = results.get("documents", [])
    metadatas = results.get("metadatas", [])
    return [
        {"content": content, "metadata": metadata or {}}
        for content, metadata in zip(documents, metadatas)
        if content
    ]

def tampilkan_halaman(page_number, max_chars=2500):
    chunks = get_page_chunks(page_number)
    if not chunks:
        print(f"Tidak menemukan teks untuk halaman {page_number}.")
        return

    text = "\n\n".join(chunk["content"] for chunk in chunks)
    print(f"ISI HALAMAN {page_number}:")
    print(text[:max_chars])
    if len(text) > max_chars:
        print("\n[Dipangkas agar output tetap ringan.]")
    print("\n> Bukti/Sumber Dokumen:")
    print(f"  - Halaman {page_number}: {len(chunks)} chunk ditemukan")

def tanyakan(pertanyaan):
    print(f"\nMemproses pertanyaan: '{pertanyaan}'...\n")

    page_number = detect_page_request(pertanyaan)
    if page_number is not None:
        tampilkan_halaman(page_number)
        return

    docs = retriever.invoke(pertanyaan)
    context_str = format_docs(docs)
    messages = prompt.format_messages(context=context_str, input=pertanyaan)
    response = llm.invoke(messages)

    print("==================== JAWABAN AI ====================")
    print(response.content)
    print("====================================================")
    print("\n> Bukti/Sumber Dokumen:")
    for doc in docs:
        clean_content = doc.page_content[:150].replace("\n", " ")
        print(f"  - Halaman {doc.metadata.get('page', 'Unknown')}: {clean_content}...")

In [7]:
# Uji cepat tanpa LLM: ambil isi halaman langsung dari metadata Chroma.
tanyakan("apa isi halamaan 10")


Memproses pertanyaan: 'apa isi halamaan 10'...

ISI HALAMAN 10:
Kegiatan 
Belaj 
ar 
Definisi dan Makna 
| 
Kebij akan Publik 
oba Anda perhatikan tentang kehidupan kita sehari-hari, baik yang menyangkut 
kehidupan ekonomi, sosial, politik, budaya, keamanan, pertahanan, lingkungan 
hidup, dan sebagainya senantiasa terkait dengan kebijakan publik di tingkat nasional, 
provinsi, dan lokal bahkan bukannya tidak mungkin di tingkat internasional. Kita tidak 
pernah bisa lepas dari berbagai masalah kebijakan (policy issues) baik yang ringan, 
sedang, berat ataupun pada aras mikro (kecil ), meso (sedang), dan makro (besar dan 
luas ). 
Bahkan disadari atau tidak perjalanan kehidupan kita ini juga banyak dipengaruhi 
oleh adanya ‘lingkungan’ dan implementasi berbagai jenis kebijakan publik pada tingkat 
lokal, nasional, regional, dan internasional. 
Demikian besarnya pengaruh kebijakan publik dalam kehidupan kita 
maka 
tidak heran banyak pihak termasuk mahasiswa ingin mempelajari dan mengkaj

In [ ]:
# Uji RAG normal: retrieval lalu jawaban dari GLM.
tanyakan("Rangkum isi halaman 125")


Memproses pertanyaan: 'Rangkum isi halaman 125'...

ISI HALAMAN 125:
ADPU4410/ MODUL 4 
Kegiatan 
Belaj ar 
Wilayah dan Ruang Lingkup 
5 
Masalah Kebijakan 
ajian kita pada Kegiatan Belajar 2 ini adalah mengenai wilayah dan ruang lingkup 
masalah kebijakan publik. 
Kajian 
ini menjadi penting karena masalah yang 
dihadapi oleh masyarakat itu ada di mana-mana, jenisnya berbeda-beda, kualitasnya 
juga berbeda-beda, persepsi masyarakat dan pemerintah terhadap masalah yang ada itu 
pun juga berbeda-beda. Ada pula masalah pribadi, masalah publik, dan masalah privat. 
Masalah mana yang kemudian menarik perhatian perumus kebijakan untuk diangkat 
menjadi masalah kebijakan (policy issue atau policy problem)? Ini adalah sesuatu yang 
rumit, Karena bukan saja masalah yang dihadapi masyarakat banyak pada umumnya 
segera diangkat oleh perumus kebijakan menjadi masalah kebijakan. Sering pula 
masalah privat yang ternyata memiliki dampak yang 
luas terhadap masyarakat juga bisa 
muncul sebagai masa

: 

In [12]:
# Coba tes dengan pertanyaan yang kemungkinan tidak ada di dokumen,
# untuk menguji agar AI tidak berhalusinasi.
tanyakan("Siapa pemenang piala dunia tahun 2022?")


Memproses pertanyaan: 'Siapa pemenang piala dunia tahun 2022?'...

==================== JAWABAN AI ====================
Saya tidak tahu.

> Bukti/Sumber Dokumen:
  - Halaman 307: masyarakat sipil dan sektor privat. Otoritas dan responsibilitas tersebar di antara  aktor-aktor yang terlibat sehingga tidak ada satu pun yang berper...
  - Halaman 40: Kunci Jawaban Tes Formatif  Tes Formatif I  1)  2)  3)  4)  5)  6)  2)  8)  9)  10)  20005565_ADPUS410_EDISI 3 1SLindb  35  A  PDOUSF RDP  Tes Formati...
